In [ ]:
# =============================================================================
# TRIAGEGEIST — AI in Emergency Triage
# Kaggle Competition Notebook
# Author: Vishwa
# =============================================================================
# STRUCTURE:
#   Phase 1 — Data loading & EDA
#   Phase 2 — Feature engineering
#   Phase 3 — Acuity prediction model (LightGBM)
#   Phase 4a — Undertriage detection
#   Phase 4b — Bias audit
#   Phase 5 — Submission generation
# =============================================================================

# ── Installs (Kaggle environment) ────────────────────────────────────────────
# !pip install lightgbm shap missingno -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Modeling
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import cohen_kappa_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.calibration import calibration_curve

# Explainability
import shap

# Missingness
import missingno as msno

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
SEED = 42

# =============================================================================
# PHASE 1 — DATA LOADING & EDA
# =============================================================================

# ── 1.1 Load all files ───────────────────────────────────────────────────────
# Adjust paths for Kaggle: /kaggle/input/triagegeist/
DATA_PATH = "/kaggle/input/competitions/triagegeist/"

train = pd.read_csv(DATA_PATH + "train.csv")
test  = pd.read_csv(DATA_PATH + "test.csv")
cc    = pd.read_csv(DATA_PATH + "chief_complaints.csv")
hist  = pd.read_csv(DATA_PATH + "patient_history.csv")
sub   = pd.read_csv(DATA_PATH + "sample_submission.csv")

print(f"Train: {train.shape} | Test: {test.shape}")
print(f"Chief complaints: {cc.shape} | History: {hist.shape}")
print(f"\nTarget distribution (triage_acuity):")
print(train['triage_acuity'].value_counts().sort_index())

# ── 1.2 Class distribution ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

train['triage_acuity'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color=['#E24B4A','#EF9F27','#378ADD','#1D9E75','#888780']
)
axes[0].set_title('ESI acuity distribution (train)')
axes[0].set_xlabel('ESI level (1=most urgent)')
axes[0].set_ylabel('Count')

# Percentage breakdown
pct = train['triage_acuity'].value_counts(normalize=True).sort_index() * 100
axes[1].pie(pct, labels=[f'ESI {i}\n({v:.1f}%)' for i,v in pct.items()],
            colors=['#E24B4A','#EF9F27','#378ADD','#1D9E75','#888780'], startangle=90)
axes[1].set_title('Acuity level proportions')
plt.tight_layout()
plt.savefig('eda_class_distribution.png', bbox_inches='tight')
plt.show()

# ── 1.3 Missingness heatmap ──────────────────────────────────────────────────
vital_cols = ['systolic_bp','diastolic_bp','heart_rate','respiratory_rate',
              'temperature_c','spo2','pain_score']

# Missingness by acuity level (pain_score uses -1 sentinel, rest are NaN)
miss_by_acuity = {}
for level in [1,2,3,4,5]:
    subset = train[train['triage_acuity'] == level].copy()
    subset['pain_score'] = subset['pain_score'].replace(-1, np.nan)
    miss_by_acuity[f'ESI {level}'] = subset[vital_cols].isnull().mean() * 100

miss_df = pd.DataFrame(miss_by_acuity).T
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(miss_df, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax, cbar_kws={'label': '% missing'})
ax.set_title('Missingness (%) by vital sign and ESI level\n(Higher ESI = lower acuity = more missingness expected)')
plt.tight_layout()
plt.savefig('eda_missingness_heatmap.png', bbox_inches='tight')
plt.show()

print("\nKey insight: missingness is clinically informative (not MCAR)")
print(miss_df.to_string())

# ── 1.4 Vital signs by acuity ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

plot_vitals = ['heart_rate','systolic_bp','respiratory_rate','spo2','temperature_c','gcs_total']
titles      = ['Heart rate (bpm)','Systolic BP (mmHg)','Resp rate (br/min)',
               'SpO₂ (%)','Temperature (°C)','GCS total']

for ax, col, title in zip(axes, plot_vitals, titles):
    data = [train[train['triage_acuity']==lvl][col].dropna() for lvl in [1,2,3,4,5]]
    bp = ax.boxplot(data, patch_artist=True, medianprops=dict(color='black', linewidth=2))
    colors = ['#E24B4A','#EF9F27','#378ADD','#1D9E75','#888780']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_xticklabels([f'ESI {i}' for i in [1,2,3,4,5]])
    ax.set_title(title)
    ax.set_ylabel(title)

plt.suptitle('Vital signs by ESI acuity level', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eda_vitals_by_acuity.png', bbox_inches='tight')
plt.show()

# ── 1.5 NEWS2 score vs acuity ────────────────────────────────────────────────
if 'news2_score' in train.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    for lvl, color in zip([1,2,3,4,5], ['#E24B4A','#EF9F27','#378ADD','#1D9E75','#888780']):
        subset = train[train['triage_acuity'] == lvl]['news2_score'].dropna()
        ax.hist(subset, bins=20, alpha=0.6, color=color, label=f'ESI {lvl}', density=True)
    ax.set_xlabel('NEWS2 score')
    ax.set_ylabel('Density')
    ax.set_title('NEWS2 score distribution by ESI level\n(NEWS2 is a clinical baseline — our model should beat this)')
    ax.legend()
    plt.tight_layout()
    plt.savefig('eda_news2_vs_acuity.png', bbox_inches='tight')
    plt.show()

    # NEWS2 correlation with acuity
    corr = train['news2_score'].corr(train['triage_acuity'])
    print(f"\nNEWS2 vs triage_acuity Pearson correlation: {corr:.3f}")
    print("(Negative = higher NEWS2 → lower ESI number = more urgent, expected)")

# ── 1.6 Chief complaint categories ──────────────────────────────────────────
if 'chief_complaint_system' in train.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    cc_counts = train.groupby(['chief_complaint_system','triage_acuity']).size().unstack(fill_value=0)
    cc_pct = cc_counts.div(cc_counts.sum(axis=1), axis=0)
    cc_pct.plot(kind='barh', stacked=True, ax=ax,
                color=['#E24B4A','#EF9F27','#378ADD','#1D9E75','#888780'])
    ax.set_xlabel('Proportion')
    ax.set_title('Acuity distribution by chief complaint system')
    ax.legend(title='ESI level', bbox_to_anchor=(1,1))
    plt.tight_layout()
    plt.savefig('eda_complaint_systems.png', bbox_inches='tight')
    plt.show()

print("\n✓ Phase 1 complete — EDA done\n")


# =============================================================================
# PHASE 2 — FEATURE ENGINEERING
# =============================================================================

def engineer_features(df, cc_df, hist_df, tfidf=None, fit_tfidf=False):
    """
    Merge all data sources and create engineered features.
    tfidf: fitted TfidfVectorizer (pass None on train, fitted object on test)
    fit_tfidf: True on train split only
    """
    df = df.copy()

    # ── 2.1 Merge chief complaints and history ────────────────────────────────
    df = df.merge(cc_df[['patient_id','chief_complaint_raw']], on='patient_id', how='left')
    df = df.merge(hist_df, on='patient_id', how='left')

    # ── 2.2 Fix pain_score sentinel ───────────────────────────────────────────
    df['pain_score'] = df['pain_score'].replace(-1, np.nan)

    # ── 2.3 Missingness indicators (clinically informative) ──────────────────
    missing_vitals = ['systolic_bp','diastolic_bp','heart_rate',
                      'respiratory_rate','temperature_c','spo2','pain_score']
    for col in missing_vitals:
        df[f'missing_{col}'] = df[col].isnull().astype(int)

    # Total missing vitals count (proxy for lower-acuity presentation)
    df['total_missing_vitals'] = df[[f'missing_{c}' for c in missing_vitals]].sum(axis=1)

    # ── 2.4 Derived clinical features ────────────────────────────────────────
    # Shock index = HR / SBP (>1.0 is clinically concerning)
    if 'shock_index' not in df.columns:
        df['shock_index'] = df['heart_rate'] / df['systolic_bp'].replace(0, np.nan)

    # Pulse pressure = SBP - DBP (narrow < 25 or wide > 60 is concerning)
    if 'pulse_pressure' not in df.columns:
        df['pulse_pressure'] = df['systolic_bp'] - df['diastolic_bp']

    # Mean arterial pressure
    if 'mean_arterial_pressure' not in df.columns:
        df['mean_arterial_pressure'] = (df['systolic_bp'] + 2 * df['diastolic_bp']) / 3

    # SpO2 < 94 flag (clinically significant hypoxia threshold)
    df['hypoxia_flag'] = (df['spo2'] < 94).astype(int)

    # GCS < 15 flag (any altered consciousness)
    df['altered_gcs'] = (df['gcs_total'] < 15).astype(int)

    # ── 2.5 Comorbidity aggregation ───────────────────────────────────────────
    hx_cols = [c for c in df.columns if c.startswith('hx_')]
    if hx_cols:
        df['total_comorbidities'] = df[hx_cols].sum(axis=1)

        # High-risk comorbidity flag (heart failure, malignancy, COPD)
        high_risk = ['hx_heart_failure','hx_malignancy','hx_copd','hx_dementia']
        present_high_risk = [c for c in high_risk if c in df.columns]
        if present_high_risk:
            df['high_risk_comorbidity'] = df[present_high_risk].max(axis=1)

    # ── 2.6 Arrival features ──────────────────────────────────────────────────
    # Ambulance arrival = high-acuity proxy
    df['ambulance_arrival'] = (df['arrival_mode'] == 'ambulance').astype(int)

    # Night/weekend shift proxy
    if 'arrival_hour' in df.columns:
        df['night_shift'] = ((df['arrival_hour'] >= 22) | (df['arrival_hour'] <= 6)).astype(int)

    # ── 2.7 TF-IDF on chief complaint text ───────────────────────────────────
    df['chief_complaint_raw'] = df['chief_complaint_raw'].fillna('unknown')

    if fit_tfidf:
        tfidf = TfidfVectorizer(max_features=50, ngram_range=(1,2),
                                stop_words='english', min_df=5)
        tfidf_matrix = tfidf.fit_transform(df['chief_complaint_raw'])
    else:
        tfidf_matrix = tfidf.transform(df['chief_complaint_raw'])

    tfidf_df = pd.DataFrame(tfidf_matrix.toarray(),
                            columns=[f'tfidf_{c}' for c in tfidf.get_feature_names_out()],
                            index=df.index)
    df = pd.concat([df, tfidf_df], axis=1)

    return df, tfidf


# Identify feature columns (everything except IDs, target, raw text, outcomes)
EXCLUDE = ['patient_id','triage_acuity','chief_complaint_raw',
           'disposition','ed_los_hours']

# Encode categoricals
CAT_COLS = ['arrival_mode','arrival_day','sex','language','insurance_type',
            'chief_complaint_system','mental_status_triage','transport_origin',
            'age_group','shift','arrival_season']

encoders = {}
def encode_cats(df):
    df = df.copy()
    for col in CAT_COLS:
        if col in df.columns:
            if col not in encoders:
                le = LabelEncoder()
                df[col] = le.fit_transform(df[col].fillna('unknown').astype(str))
                encoders[col] = le
            else:
                df[col] = encoders[col].transform(df[col].fillna('unknown').astype(str))
    return df

print("✓ Phase 2 functions defined\n")


# =============================================================================
# PHASE 3 — ACUITY PREDICTION MODEL
# =============================================================================

# ── 3.1 Prepare data ──────────────────────────────────────────────────────────
train_fe, fitted_tfidf = engineer_features(train, cc, hist, fit_tfidf=True)
test_fe, _             = engineer_features(test,  cc, hist, tfidf=fitted_tfidf, fit_tfidf=False)

train_fe = encode_cats(train_fe)
test_fe  = encode_cats(test_fe)

feature_cols = [c for c in train_fe.columns
                if c not in EXCLUDE
                and train_fe[c].dtype in ['int64','float64','int32','float32']]

X = train_fe[feature_cols].values
y = train_fe['triage_acuity'].values - 1  # 0-indexed for LightGBM

X_test = test_fe[feature_cols].values

print(f"Features: {len(feature_cols)}")
print(f"X shape: {X.shape} | y shape: {y.shape}")

# ── 3.2 Baseline 1: always predict mode ──────────────────────────────────────
from scipy.stats import mode
mode_pred = np.full(len(y), mode(y).mode)
baseline_kappa = cohen_kappa_score(y, mode_pred, weights='quadratic')
print(f"\nBaseline (mode) QWK: {baseline_kappa:.4f}")

# ── 3.3 Baseline 2: NEWS2 threshold ──────────────────────────────────────────
if 'news2_score' in train_fe.columns:
    def news2_to_esi(score):
        # NEWS2 ≥7 → ESI 1, 5-6 → ESI 2, 3-4 → ESI 3, 1-2 → ESI 4, 0 → ESI 5
        if score >= 7: return 0
        elif score >= 5: return 1
        elif score >= 3: return 2
        elif score >= 1: return 3
        else: return 4

    news2_pred = train_fe['news2_score'].fillna(0).apply(news2_to_esi).values
    news2_kappa = cohen_kappa_score(y, news2_pred, weights='quadratic')
    print(f"Baseline (NEWS2 thresholds) QWK: {news2_kappa:.4f}")

# ── 3.4 LightGBM with stratified 5-fold CV ───────────────────────────────────
LGBM_PARAMS = {
    'objective': 'multiclass',
    'num_class': 5,
    'metric': 'multi_logloss',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'max_depth': -1,
    'min_child_samples': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'class_weight': 'balanced',   # handles class imbalance (like fraud detection!)
    'verbose': -1,
    'seed': SEED,
}

SKF = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

oof_preds   = np.zeros((len(X), 5))
test_preds  = np.zeros((len(X_test), 5))
fold_kappas = []
models      = []

for fold, (train_idx, val_idx) in enumerate(SKF.split(X, y)):
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    dtrain = lgb.Dataset(X_tr, label=y_tr, feature_name=feature_cols)
    dval   = lgb.Dataset(X_val, label=y_val, feature_name=feature_cols)

    model = lgb.train(
        LGBM_PARAMS,
        dtrain,
        num_boost_round=1000,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(200)]
    )

    val_prob  = model.predict(X_val)
    val_class = np.argmax(val_prob, axis=1)

    kappa = cohen_kappa_score(y_val, val_class, weights='quadratic')
    fold_kappas.append(kappa)
    print(f"Fold {fold+1} QWK: {kappa:.4f}")

    oof_preds[val_idx] = val_prob
    test_preds += model.predict(X_test) / 5
    models.append(model)

overall_kappa = cohen_kappa_score(y, np.argmax(oof_preds, axis=1), weights='quadratic')
print(f"\n{'='*50}")
print(f"OOF QWK: {overall_kappa:.4f}  (mean fold: {np.mean(fold_kappas):.4f})")
print(f"Improvement over mode baseline: +{overall_kappa - baseline_kappa:.4f}")
if 'news2_kappa' in dir():
    print(f"Improvement over NEWS2 baseline: +{overall_kappa - news2_kappa:.4f}")

# ── 3.5 Confusion matrix ─────────────────────────────────────────────────────
oof_classes = np.argmax(oof_preds, axis=1)
cm = confusion_matrix(y, oof_classes)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=[f'ESI {i+1}' for i in range(5)],
            yticklabels=[f'ESI {i+1}' for i in range(5)])
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'OOF confusion matrix (QWK={overall_kappa:.4f})')
plt.tight_layout()
plt.savefig('model_confusion_matrix.png', bbox_inches='tight')
plt.show()

# ── 3.6 SHAP feature importance ──────────────────────────────────────────────
explainer = shap.TreeExplainer(models[0])
shap_sample = X[:2000]

# For multiclass LightGBM, shap_values returns shape (n_samples, n_features, n_classes)
# or a list of (n_samples, n_features) — handle both
raw_shap = explainer.shap_values(shap_sample)

if isinstance(raw_shap, list):
    # List of arrays, one per class: each is (2000, 122)
    mean_shap = np.mean([np.abs(sv) for sv in raw_shap], axis=0)  # (2000, 122)
    mean_shap = mean_shap.mean(axis=0)  # (122,)
else:
    # 3D array: (2000, 122, 5)
    mean_shap = np.abs(raw_shap).mean(axis=2).mean(axis=0)  # (122,)

print(f"raw_shap type: {type(raw_shap)}")
if isinstance(raw_shap, list):
    print(f"List length: {len(raw_shap)}, each shape: {raw_shap[0].shape}")
else:
    print(f"Array shape: {raw_shap.shape}")
print(f"mean_shap shape: {mean_shap.shape}")
print(f"feature_cols length: {len(feature_cols)}")

assert len(mean_shap) == len(feature_cols), \
    f"Shape mismatch: {len(mean_shap)} shap values vs {len(feature_cols)} features"

shap_df = pd.DataFrame({'feature': feature_cols, 'importance': mean_shap})
shap_df = shap_df.sort_values('importance', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(shap_df['feature'][::-1], shap_df['importance'][::-1], color='#378ADD')
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('Top 20 features by SHAP importance\n(averaged across all ESI classes)')
plt.tight_layout()
plt.savefig('model_shap_importance.png', bbox_inches='tight')
plt.show()

# =============================================================================
# PHASE 4a — UNDERTRIAGE DETECTION
# =============================================================================
# Definition: patient assigned ESI 4 or 5 (low acuity) but actually needed
# significant care (admitted or long LOS). These are the dangerous misses.

if 'disposition' in train_fe.columns and 'ed_los_hours' in train_fe.columns:

    LOS_THRESHOLD = 6.0  # hours — above this suggests higher-than-expected severity

    train_fe['undertriage_flag'] = (
        (train_fe['triage_acuity'].isin([4, 5])) &
        ((train_fe['disposition'].str.lower().str.contains('admit', na=False)) |
         (train_fe['ed_los_hours'] > LOS_THRESHOLD))
    ).astype(int)

    n_undertriaged = train_fe['undertriage_flag'].sum()
    pct_undertriaged = n_undertriaged / len(train_fe) * 100
    print(f"Undertriaged patients: {n_undertriaged} ({pct_undertriaged:.1f}% of all patients)")
    print(f"As % of ESI 4/5 patients: "
          f"{n_undertriaged / train_fe['triage_acuity'].isin([4,5]).sum() * 100:.1f}%")

    # Train secondary classifier on full dataset
    X_ut = train_fe[feature_cols].values
    y_ut = train_fe['undertriage_flag'].values

    # Use LightGBM binary classifier
    UT_PARAMS = {
        'objective': 'binary',
        'metric': 'auc',
        'learning_rate': 0.05,
        'num_leaves': 31,
        'class_weight': 'balanced',
        'verbose': -1,
        'seed': SEED,
    }

    oof_ut_probs = np.zeros(len(X_ut))
    for fold, (tr_idx, val_idx) in enumerate(SKF.split(X_ut, y_ut)):
        d_tr = lgb.Dataset(X_ut[tr_idx], label=y_ut[tr_idx])
        d_val = lgb.Dataset(X_ut[val_idx], label=y_ut[val_idx])
        ut_model = lgb.train(UT_PARAMS, d_tr, 500,
                             valid_sets=[d_val],
                             callbacks=[lgb.early_stopping(50), lgb.log_evaluation(200)])
        oof_ut_probs[val_idx] = ut_model.predict(X_ut[val_idx])

    from sklearn.metrics import roc_auc_score, average_precision_score
    ut_auc = roc_auc_score(y_ut, oof_ut_probs)
    ut_ap  = average_precision_score(y_ut, oof_ut_probs)
    print(f"\nUndertriage classifier OOF AUC: {ut_auc:.4f}")
    print(f"Undertriage classifier OOF AP:  {ut_ap:.4f}")

    # Plot risk score distribution
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(oof_ut_probs[y_ut==0], bins=50, alpha=0.6, label='Correctly triaged', color='#378ADD', density=True)
    ax.hist(oof_ut_probs[y_ut==1], bins=50, alpha=0.6, label='Undertriaged (hidden high risk)', color='#E24B4A', density=True)
    ax.set_xlabel('Undertriage risk score')
    ax.set_ylabel('Density')
    ax.set_title(f'Undertriage risk score distribution\n(AUC={ut_auc:.3f}, AP={ut_ap:.3f})')
    ax.legend()
    plt.tight_layout()
    plt.savefig('undertriage_risk_distribution.png', bbox_inches='tight')
    plt.show()

print("\n✓ Phase 4a complete — undertriage analysis done\n")


# =============================================================================
# PHASE 4b — BIAS AUDIT
# =============================================================================
# Examine whether model errors are systematically skewed by demographic group.
# This mirrors disparate impact analysis in fraud detection.

BIAS_COLS = ['sex', 'age_group', 'insurance_type', 'language']

# Use original (pre-encoded) values from train merged with OOF predictions
train_with_preds = train_fe.copy()
train_with_preds['oof_pred'] = oof_classes
train_with_preds['oof_pred_esi'] = oof_classes + 1  # back to 1-5 scale
train_with_preds['true_acuity'] = y + 1

# Absolute error in ESI levels (0=correct, 1=off by one, etc.)
train_with_preds['abs_error'] = abs(train_with_preds['oof_pred_esi'] - train_with_preds['true_acuity'])

# Undertriage at model level: predicted lower acuity than actual
# (e.g. predicted ESI 4 but true ESI 2 = missed high-acuity)
train_with_preds['model_undertriage'] = (
    train_with_preds['oof_pred_esi'] > train_with_preds['true_acuity'] + 1
).astype(int)

print("Bias audit results:")
print("=" * 60)

bias_results = {}
for col in BIAS_COLS:
    if col not in train_with_preds.columns:
        continue

    # Re-decode if label-encoded
    if col in encoders:
        original_col = encoders[col].inverse_transform(
            train_with_preds[col].fillna(-1).astype(int).clip(0)
        )
        train_with_preds[f'{col}_orig'] = original_col
        use_col = f'{col}_orig'
    else:
        use_col = col

    group_stats = train_with_preds.groupby(use_col).agg(
        n=('abs_error', 'count'),
        mean_abs_error=('abs_error', 'mean'),
        kappa=('true_acuity', lambda x: cohen_kappa_score(
            x, train_with_preds.loc[x.index, 'oof_pred_esi'], weights='quadratic'
        ) if len(x) > 10 else np.nan),
        undertriage_rate=('model_undertriage', 'mean')
    ).round(4)

    print(f"\n{col.upper()}:")
    print(group_stats.to_string())
    bias_results[col] = group_stats

    # Bar chart of mean absolute error by group
    fig, ax = plt.subplots(figsize=(8, 3))
    group_stats['mean_abs_error'].sort_values().plot(kind='barh', ax=ax, color='#7F77DD')
    ax.set_xlabel('Mean absolute ESI error')
    ax.set_title(f'Prediction error by {col}\n(higher = more error = potential bias)')
    ax.axvline(train_with_preds['abs_error'].mean(), color='#E24B4A',
               linestyle='--', label='Overall mean')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'bias_{col}.png', bbox_inches='tight')
    plt.show()

print("\n✓ Phase 4b complete — bias audit done\n")


# =============================================================================
# PHASE 5 — SUBMISSION GENERATION
# =============================================================================

final_preds = np.argmax(test_preds, axis=1) + 1  # back to 1-5 ESI scale

submission = pd.DataFrame({
    'patient_id': test['patient_id'],
    'triage_acuity': final_preds
})

# Verify format matches sample submission
assert set(submission.columns) == set(sub.columns), "Column mismatch with sample submission"
assert submission['triage_acuity'].between(1, 5).all(), "Predictions out of valid range"

submission.to_csv('submission.csv', index=False)
print(f"Submission saved: {submission.shape}")
print(submission['triage_acuity'].value_counts().sort_index())

# ── Summary card ─────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("TRIAGEGEIST SUBMISSION SUMMARY")
print("="*60)
print(f"Model:               LightGBM 5-fold CV")
print(f"Features:            {len(feature_cols)} (vitals + demographics + comorbidities + TF-IDF)")
print(f"OOF QWK:             {overall_kappa:.4f}")
print(f"vs mode baseline:    +{overall_kappa - baseline_kappa:.4f}")
if 'news2_kappa' in dir():
    print(f"vs NEWS2 baseline:   +{overall_kappa - news2_kappa:.4f}")
if 'ut_auc' in dir():
    print(f"Undertriage AUC:     {ut_auc:.4f}")
print(f"Bias cols audited:   {', '.join(BIAS_COLS)}")
print("="*60)